In [1]:
import numpy as np, jax.numpy as jnp
from jax import random
from scipy.stats import norm

def importance_sampling(
    f, p_target, proposal_mean, proposal_variance,
    sample_size=1000, seed=0, *,
    use_f=False,            # <-- NEW: default keeps old behavior
    self_normalized=False   # <-- optional, for unnormalized targets
):
    """
    Importance sampling with a Normal proposal q = N(mean, variance).

    Default (use_f=False): estimates E_p[Z] — SAME as your original function.
    If use_f=True: estimates E_p[f(Z)].
    If self_normalized=True: returns sum(w*f(z))/sum(w) instead of mean(w*f(z)).

    Returns a Python float.
    """
    key = random.PRNGKey(seed)
    q_pdf = lambda z: norm.pdf(z, proposal_mean, np.sqrt(proposal_variance))

    # z ~ q
    z = proposal_mean + jnp.sqrt(proposal_variance) * random.normal(key, shape=(sample_size,))

    # importance weights w = p(z)/q(z)
    w = p_target(z) / (q_pdf(z) + 1e-300)  # small epsilon for safety

    # what to integrate: default is z (old behavior); otherwise f(z)
    g = z if not use_f else f(z)

    est = np.sum(w * g) / np.sum(w) if self_normalized else np.mean(w * g)
    return float(est)


In [2]:
p = lambda z: norm.pdf(z, 0.0, 1.0)              # target: N(0,1)
est_mean = importance_sampling(
    f=lambda z: z, p_target=p,
    proposal_mean=1.0, proposal_variance=4.0,
    sample_size=50_000, seed=0, use_f=False      # <- default; same as before
)
print(est_mean)  # ~ 0


0.00045538603444583714


In [9]:
est_second = importance_sampling(
    f=lambda z: z**1, p_target=p,
    proposal_mean=1.0, proposal_variance=4.0,
    sample_size=50_000, seed=1, use_f=True
)
est_var = est_second - est_mean**2
print(est_second)  # ~ 1
print(est_var)     # ~ 1


-0.0017887450521811843
-0.0017889524286215527


In [4]:
def estimate_mean_and_variance(p_target, mean_q, var_q, n=50_000, seed=0, self_norm=False):
    m = importance_sampling(lambda z: z,   p_target, mean_q, var_q, n, seed,   use_f=True, self_normalized=self_norm)
    s2 = importance_sampling(lambda z: z**2, p_target, mean_q, var_q, n, seed+1, use_f=True, self_normalized=self_norm)
    return m, max(0.0, s2 - m**2)  # clamp tiny negatives from MC noise

m, v = estimate_mean_and_variance(p, 1.0, 4.0)
print(m, v)


0.00045538603444583714 1.0037513089657226
